# Machine Learning-Based Buyer Segmentation and Investment Profiling for Real Estate Market Intelligence

**Project starter notebook — Parcl project brief**

> **Dataset note:** The included CSV is synthetic practice data created to match the field names in the project brief. It is not actual Parcl customer data. Replace it with an authorized company dataset before making company-specific claims.

## Objectives
- Inspect and clean buyer data.
- Engineer buyer age from date of birth.
- Encode categorical features and scale numeric features.
- Compare K-Means and hierarchical clustering.
- Evaluate K-Means cluster counts using the Elbow Method and Silhouette Score.
- Summarize segments for a business audience.


In [ ]:
# 1. Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

pd.set_option("display.max_columns", None)


## 2. Load the dataset
In Google Colab, upload `real_estate_buyer_synthetic_dataset.csv` when prompted.

In [ ]:
# Google Colab upload
from google.colab import files
uploaded = files.upload()

# If running locally, replace this with:
# df = pd.read_csv("real_estate_buyer_synthetic_dataset.csv")

csv_name = next(name for name in uploaded if name.lower().endswith(".csv"))
df = pd.read_csv(csv_name)
print("Shape:", df.shape)
display(df.head())


In [ ]:
# 3. Initial data-quality checks
print("Column types:")
print(df.dtypes)
print("\\nMissing values:")
display(df.isna().sum().to_frame("missing_count"))
print("Exact duplicate rows:", df.duplicated().sum())
print("Unique client IDs:", df["client_id"].nunique())


## 4. Clean and prepare features
We keep `client_id` out of the model because it is an identifier, not a buyer behaviour feature. Age is calculated from `date_of_birth`.

In [ ]:
# Remove exact duplicate rows and duplicate client IDs (keep first record)
df = df.drop_duplicates().copy()
df = df.drop_duplicates(subset=["client_id"], keep="first").copy()

# Parse date of birth and derive age
df["date_of_birth"] = pd.to_datetime(df["date_of_birth"], errors="coerce")
reference_date = pd.Timestamp.today().normalize()
df["age"] = ((reference_date - df["date_of_birth"]).dt.days / 365.25).round()

# Keep realistic age range; invalid values become missing and will be imputed
df.loc[~df["age"].between(18, 100), "age"] = np.nan

# Remove fields not used as model inputs
feature_cols = [
    "client_type", "gender", "country", "region", "age",
    "acquisition_purpose", "loan_applied", "referral_channel",
    "satisfaction_score"
]
model_df = df[feature_cols].copy()

categorical_features = [
    "client_type", "gender", "country", "region",
    "acquisition_purpose", "loan_applied", "referral_channel"
]
numeric_features = ["age", "satisfaction_score"]

# Fill missing values, one-hot encode categories, scale numeric fields
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipe, numeric_features),
    ("categorical", categorical_pipe, categorical_features)
])

X = preprocessor.fit_transform(model_df)
print("Prepared model matrix shape:", X.shape)


## 5. Explore the data (EDA)

In [ ]:
# Buyer counts by purpose and country
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
df["acquisition_purpose"].value_counts(dropna=False).plot(kind="bar", ax=axes[0], title="Buyers by acquisition purpose")
df["country"].value_counts(dropna=False).plot(kind="bar", ax=axes[1], title="Buyers by country")
axes[0].set_xlabel("Purpose")
axes[1].set_xlabel("Country")
plt.tight_layout()
plt.show()

display(df.groupby("acquisition_purpose", dropna=False).agg(
    buyer_count=("client_id", "count"),
    mean_satisfaction=("satisfaction_score", "mean"),
    loan_application_rate=("loan_applied", lambda s: (s == "Yes").mean())
).round(2))


## 6. Choose the number of clusters
The Elbow Method and Silhouette Score are guides, not automatic proof that a particular cluster count is best.

In [ ]:
inertias, silhouettes = [], []
candidate_ks = range(2, 8)
for k in candidate_ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(candidate_ks), inertias, marker="o")
axes[0].set_title("Elbow Method")
axes[0].set_xlabel("Number of clusters (k)")
axes[0].set_ylabel("Inertia")
axes[1].plot(list(candidate_ks), silhouettes, marker="o")
axes[1].set_title("Silhouette Score")
axes[1].set_xlabel("Number of clusters (k)")
axes[1].set_ylabel("Score")
plt.tight_layout()
plt.show()

score_table = pd.DataFrame({"k": list(candidate_ks), "inertia": inertias, "silhouette_score": silhouettes})
display(score_table.round(3))
best_k = int(score_table.loc[score_table["silhouette_score"].idxmax(), "k"])
print("Best k by silhouette score in this run:", best_k)


## 7. Fit K-Means and hierarchical clustering

In [ ]:
# Use the best silhouette score as a reproducible starting choice.
# Review the plots and business usefulness before finalizing k.
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df["kmeans_cluster"] = kmeans.fit_predict(X).astype(str)

hierarchical = AgglomerativeClustering(n_clusters=best_k, linkage="ward")
df["hierarchical_cluster"] = hierarchical.fit_predict(X).astype(str)

print("K-Means silhouette:", round(silhouette_score(X, df["kmeans_cluster"].astype(int)), 3))
print("Hierarchical silhouette:", round(silhouette_score(X, df["hierarchical_cluster"].astype(int)), 3))
display(df[["client_id", "country", "acquisition_purpose", "loan_applied", "satisfaction_score", "kmeans_cluster"]].head(10))


## 8. Interpret clusters
Cluster labels are arbitrary. Describe each cluster from its profile rather than assuming a number always means the same buyer type.

In [ ]:
cluster_profile = df.groupby("kmeans_cluster").agg(
    buyer_count=("client_id", "count"),
    median_age=("age", "median"),
    investment_share=("acquisition_purpose", lambda s: (s == "Investment").mean()),
    loan_application_share=("loan_applied", lambda s: (s == "Yes").mean()),
    mean_satisfaction=("satisfaction_score", "mean"),
    corporate_share=("client_type", lambda s: (s == "Corporate").mean())
).round(2)

display(cluster_profile)
cluster_profile.to_csv("cluster_profile.csv")
df.to_csv("buyer_segments_output.csv", index=False)
print("Saved cluster_profile.csv and buyer_segments_output.csv")


## 9. Business recommendations — write from actual results
Use the cluster profile above to write evidence-based recommendations. For example:
- If a segment has a high loan-application share, offer clear financing information and loan eligibility support.
- If a segment is primarily investment-oriented, provide comparable-property information and transparent return/risk assumptions.
- If satisfaction is lower in a segment, investigate service issues before increasing marketing spend.
- Compare acquisition channels by segment before reallocating campaign budgets.

Do not claim that these recommendations increase sales unless this is measured with further testing.

## Limitations
- The supplied dataset is synthetic and does not represent actual customers.
- The available fields do not include property prices, income, budget, transaction value, or actual investment returns.
- Clusters reveal patterns in the supplied features; they do not prove causal relationships.
- Treat demographic fields responsibly and avoid using protected attributes to make unfair decisions.
